# 01 — Project setup

Walk-through of getting CHARMPheno running locally. Target audience: a contributor who has just cloned the repo and has no other context.

## Prerequisites

- Python 3.10–3.12.
- Java 17 available (local Spark requires it — Java 23+ has Arrow compatibility issues in PySpark 3.5).
  - macOS: `brew install openjdk@17`
  - Linux: `apt install openjdk-17-jdk` or equivalent.
- Poetry 2.x (`pipx install poetry`).


## Repository layout

Top-level:

- `docs/architecture/` — the living architectural vision. Start here if you want the *why*.
- `docs/decisions/` — ADRs. One per significant organizational choice.
- `docs/superpowers/specs/` and `docs/superpowers/plans/` — design documents and implementation plans.
- `spark-vi/` — the reusable, domain-agnostic distributed-VI framework.
- `charmpheno/` — the clinical specialization. Depends on `spark-vi`.
- `analysis/` — runnable end-to-end scripts.
- `notebooks/tutorials/` — exposition notebooks (you are here).
- `scripts/` — data fetch, simulator, dev helpers.
- `AGENTS.md` — orientation for LLM-based coding agents.


## Install everything

```bash
make install-dev
```

This does three things:

1. Installs the root venv (`.venv/`) used by scripts and pre-commit.
2. Installs `spark-vi`'s own venv (`spark-vi/.venv/`).
3. Installs `charmpheno`'s own venv (`charmpheno/.venv/`), with `spark-vi` installed editable inside it.

During dev, changes to `spark-vi` flow through to `charmpheno` immediately without reinstall.


## Fetch reference β and generate synthetic data

```bash
make data
```

This fetches the LDA β from Hugging Face (streaming; ~15 MB after top-K filter) into `data/cache/lda_beta_topk.parquet`, then generates a synthetic OMOP parquet under `data/simulated/`.

`data/` is `.gitignore`d. Nothing generated here ever reaches the repo.


## Run the tests

```bash
make test          # unit only, <10s, your default iteration loop
make test-all      # + integration tests (@slow), a few minutes
make test-cluster  # cluster-only tests, manual on a Dataproc setup
```

The default `make test` is deliberately fast. If your change touches an integration surface, run `make test-all` before pushing.


## End-to-end smoke

After `make data`, run:

```bash
cd charmpheno && poetry run python ../analysis/local/fit_charmpheno_local.py \
    --input ../data/simulated/omop_N10000_seed0.parquet \
    --output ../tmp/result \
    --max-iterations 10
```

This runs the data → VIRunner (CountingModel placeholder) → saved artifact path. It confirms plumbing — real HDP math lives in a follow-on spec.


## Packaging artifacts

```bash
make build      # builds wheels + sdists for both packages
make zip        # builds flat zips for both packages
```

Both artifact types are required. Wheels go to `pip install` targets; zips go to Spark executors via `sc.addPyFile('…/pkg.zip')` or `--py-files`.

## Deploying to a cloud notebook environment (appendix)

The canonical pattern — deployed to a GCS-backed Dataproc + Jupyter workbench — is:

1. After `make build` + `make zip` on a dev machine, upload `spark-vi/dist/*.whl`, `spark-vi/dist/spark_vi.zip`, `charmpheno/dist/*.whl`, and `charmpheno/dist/charmpheno.zip` to a workspace GCS bucket (`gs://<bucket>/pkgs/`).
2. In the notebook environment: `pip install gs://<bucket>/pkgs/<wheels>` for the driver; `sc.addPyFile('gs://<bucket>/pkgs/<zip>')` for each package for executors.
3. Import and run — code developed locally runs unchanged.

This completes the basic orientation. Next: see `docs/architecture/` for the research design and framework architecture, and individual design specs under `docs/superpowers/specs/` for per-feature context.
